In [ ]:
# ========== 在 Jupyter Notebook 中运行 ==========
import os
import re
import csv
from pathlib import Path
import pandas as pd  # 可选，用于美观展示表格
from IPython.display import display, HTML

# ========== 配置 ==========
TRAIN_ROOT = r"E:\guanxueying\graduation_project\yolo\tobacco\formal\model"
# =========================

def get_model_size_from_yaml(yaml_path):
    if not os.path.exists(yaml_path):
        return "N/A"
    try:
        with open(yaml_path, 'r', encoding='utf-8') as f:
            content = f.read()
        scale_pattern = r'scale:\s*(["\']?)([a-zA-Z]+)\1'
        match = re.search(scale_pattern, content)
        if match:
            return match.group(2)
        depth_match = re.search(r'depth_multiple:\s*([0-9.]+)', content)
        width_match = re.search(r'width_multiple:\s*([0-9.]+)', content)
        if depth_match and width_match:
            depth = float(depth_match.group(1))
            width = float(width_match.group(1))
            mapping = {
                (0.33, 0.25): "n",
                (0.33, 0.50): "s",
                (0.67, 0.75): "m",
                (1.00, 1.00): "l",
                (1.33, 1.25): "x"
            }
            return mapping.get((depth, width), f"custom({depth},{width})")
        return "unknown"
    except Exception as e:
        return f"parse_error: {e}"

def get_input_size_from_yaml(yaml_path):
    if not os.path.exists(yaml_path):
        return "N/A"
    try:
        with open(yaml_path, 'r', encoding='utf-8') as f:
            content = f.read()
        pattern = r'imgsz:\s*(\d+(?:,\s*\d+)?|\[(?:\d+,\s*)+\d+\])'
        match = re.search(pattern, content)
        if match:
            val = match.group(1)
            if ',' in val:
                sizes = [s.strip() for s in val.replace('[', '').replace(']', '').split(',')]
                return f"{sizes[0]}x{sizes[1]}" if len(sizes) >= 2 else val
            else:
                return f"{val}x{val}"
        return "640x640 (default)"
    except Exception:
        return "N/A"

def extract_best_metrics(results_csv_path):
    """
    优先级：mask mAP50-95(M) > mask mAP50(M) > box mAP50(B)
    返回 (best_map50, best_map50_95, best_epoch)，均以最优 mask mAP50-95 对应的 epoch 为准
    """
    if not os.path.exists(results_csv_path):
        return "N/A", "N/A", "N/A"
    try:
        df = pd.read_csv(results_csv_path)
        df.columns = df.columns.str.strip()  # 防止列名含空格

        # ---------- 定位各列 ----------
        # 优先级1：mask mAP50-95
        MASK_MAP95_CANDIDATES = [
            'metrics/mAP50-95(M)',
            'metrics/map50-95(m)',
            'metrics/mAP50_95(M)',
        ]
        # 优先级2：mask mAP50
        MASK_MAP50_CANDIDATES = [
            'metrics/mAP50(M)',
            'metrics/map50(m)',
        ]
        # 回退：box mAP50
        BOX_MAP50_CANDIDATES = [
            'metrics/mAP50(B)',
            'metrics/map50(b)',
        ]

        def find_col(candidates, df):
            for c in candidates:
                for col in df.columns:
                    if col.lower() == c.lower():
                        return col
            return None

        mask_map95_col = find_col(MASK_MAP95_CANDIDATES, df)
        mask_map50_col = find_col(MASK_MAP50_CANDIDATES, df)
        box_map50_col  = find_col(BOX_MAP50_CANDIDATES, df)

        # ---------- 按优先级选定排序列 ----------
        if mask_map95_col:
            sort_col = mask_map95_col          # ✅ 首选：mask mAP50-95
        elif mask_map50_col:
            sort_col = mask_map50_col          # 回退1：mask mAP50
        elif box_map50_col:
            sort_col = box_map50_col           # 回退2：box mAP50
        else:
            return "N/A(no metric col)", "N/A", "N/A"

        # ---------- 找最优 epoch ----------
        best_idx   = df[sort_col].idxmax()
        best_epoch = df.loc[best_idx, 'epoch'] if 'epoch' in df.columns else best_idx

        # ---------- 读取该 epoch 的 map50 / map50-95 ----------
        # 报告列：优先 mask，无则 box
        report_map50_col = mask_map50_col or box_map50_col
        report_map95_col = mask_map95_col

        best_map50 = (
            f"{df.loc[best_idx, report_map50_col]:.4f}"
            if report_map50_col else "N/A"
        )
        best_map95 = (
            f"{df.loc[best_idx, report_map95_col]:.4f}"
            if report_map95_col else "N/A"
        )

        return best_map50, best_map95, str(int(best_epoch))

    except Exception as e:
        return f"error: {e}", "N/A", "N/A"

# 收集数据
data = []
folders = [f for f in os.listdir(TRAIN_ROOT) if os.path.isdir(os.path.join(TRAIN_ROOT, f))]

for folder in folders:
    folder_path = os.path.join(TRAIN_ROOT, folder)
    # 查找 yaml 文件
    yaml_path = None
    for yaml_file in ['args.yaml', 'opt.yaml', 'train.yaml', 'yolo11.yaml']:
        temp = os.path.join(folder_path, yaml_file)
        if os.path.exists(temp):
            yaml_path = temp
            break
    if not yaml_path:
        for f in os.listdir(folder_path):
            if f.endswith('.yaml'):
                yaml_path = os.path.join(folder_path, f)
                break

    results_path = os.path.join(folder_path, 'results.csv')
    model_size = get_model_size_from_yaml(yaml_path) if yaml_path else "N/A"
    imgsz = get_input_size_from_yaml(yaml_path) if yaml_path else "N/A"
    map50, map95, epoch = extract_best_metrics(results_path)

    data.append({
        "训练文件夹": folder,
        "模型类型": model_size,
        "输入尺寸": imgsz,
        "最佳 mAP50": map50,
        "最佳 mAP50-95": map95,
        "最佳 epoch": epoch
    })

# 用 DataFrame 显示
df_result = pd.DataFrame(data)
display(df_result)

# 可选：保存为 CSV 文件
df_result.to_csv("model_comparison_report.csv", index=False, encoding='utf-8-sig')
print("\n报告已保存为 model_comparison_report.csv")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# ========== 数据 ==========
data = {
    's': {'1920': 0.6125, '2560': 0.6395},
    'm': {'1920': 0.6064, '2560': 0.6685},
    'l': {'1920': 0.6154, '2560': 0.6716},
}

model_order = ['s', 'm', 'l']
sizes = ['1920', '2560']

# 参考例图配色（蓝/橙）
colors = {
    '1920': '#1f77b4',   # 蓝
    '2560': '#ff7f0e',   # 橙
}
markers = {
    '1920': 'o',
    '2560': 'o',
}

# ========== 绘图 ==========
fig, ax = plt.subplots(figsize=(7, 5))

x = np.arange(len(model_order))

for sz in sizes:
    y = [data[m][sz] for m in model_order]
    ax.plot(
        x, y,
        color=colors[sz],
        marker=markers[sz],
        linewidth=2,
        markersize=8,
        label=f'{sz}×{sz}',
    )
    # 数据标签
    for xi, yi in zip(x, y):
        ax.annotate(
            f'{yi:.4f}',
            xy=(xi, yi),
            xytext=(0, 10),
            textcoords='offset points',
            ha='center',
            fontsize=9,
            color=colors[sz],
        )

# ========== 坐标轴 ==========
ax.set_xticks(x)
ax.set_xticklabels([f'YOLO11{m}' for m in model_order], fontsize=11)
ax.set_xlabel('Model Variant', fontsize=12)
ax.set_ylabel('Val Mask mAP@0.50:0.95', fontsize=12)
ax.set_title('Model Variant vs Val Mask mAP50-95', fontsize=13)

# Y轴范围留出标签空间
y_all = [data[m][sz] for m in model_order for sz in sizes]
ax.set_ylim(min(y_all) - 0.015, max(y_all) + 0.025)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

ax.grid(axis='y', linestyle='--', alpha=0.6)
ax.legend(title='Input Size', fontsize=10, title_fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -*- coding: utf-8 -*-
"""YOLO11 训练信息查看器（交互式，适用于 Jupyter）"""

import sys
from pathlib import Path

# 确保 ultralytics 已安装：!pip install ultralytics
try:
    from ultralytics import YOLO
except ImportError:
    print("错误：未安装 ultralytics，请先执行：!pip install ultralytics")
    sys.exit(1)

# ========== 1. 交互式输入模型文件路径 ==========
while True:
    model_path = input("请输入 YOLO11 模型文件路径（例如 best.pt）：").strip().strip('"').strip("'")
    if not model_path:
        print("路径不能为空，请重新输入。")
        continue
    p = Path(model_path)
    if not p.exists():
        print(f"文件不存在：{model_path}，请重新输入。")
        continue
    if p.suffix.lower() not in ('.pt', '.pth'):
        print(f"请提供 .pt 或 .pth 文件（当前后缀：{p.suffix}）")
        continue
    break

print(f"\n正在加载模型：{p} ...")

# ========== 2. 加载模型并提取信息 ==========
try:
    model = YOLO(model_path)
    ckpt = model.ckpt  # 训练时保存的完整信息
    train_args = ckpt.get('train_args', {})
    if not train_args:
        # 备用方案：从 model.overrides 获取部分信息
        train_args = model.overrides if hasattr(model, 'overrides') else {}
    
    # 模型变体：通常保存在 ckpt['model'].yaml 的 'names' 或模型文件名推断
    # YOLO11 常用变体：yolo11n, yolo11s, yolo11m, yolo11l, yolo11x
    # 简单从模型文件名字推断（如 yolo11n.pt 或 best.pt）
    model_name = p.stem
    # 也可以从 model.model.yaml.get('nc') 等特征推测，但直接读取原配置更准确
    yaml_cfg = ckpt.get('model', {}).get('yaml', {}) if isinstance(ckpt.get('model'), dict) else {}
    if 'yaml' in ckpt:
        # 不同版本结构不同
        cfg = ckpt['yaml'] if isinstance(ckpt['yaml'], dict) else {}
    else:
        cfg = {}
    
    # 尝试获取具体的架构标识（通过 model.model.args 中的 'model' 键）
    arch = None
    if hasattr(model, 'model') and hasattr(model.model, 'args'):
        arch = model.model.args.get('model', None)
    if not arch and 'model' in train_args:
        arch = train_args['model']
    if not arch:
        arch = "未知（请从文件名或配置推测）"
    
    # 输入尺寸（imgsz）
    imgsz = train_args.get('imgsz', None)
    if imgsz is None and hasattr(model, 'model') and hasattr(model.model, 'stride'):
        # 默认 imgsz 可能为 640
        imgsz = 640
    elif isinstance(imgsz, list):
        imgsz = imgsz[0]  # 通常取第一个
    
    # 超参数（hyp）
    hyp_keys = ['lr0', 'lrf', 'momentum', 'weight_decay', 'warmup_epochs', 'warmup_momentum',
                'box', 'cls', 'dfl', 'hsv_h', 'hsv_s', 'hsv_v', 'degrees', 'translate', 'scale',
                'shear', 'perspective', 'flipud', 'fliplr', 'mosaic', 'mixup', 'copy_paste']
    hyp = {k: train_args.get(k) for k in hyp_keys if k in train_args}
    
    # 额外训练信息
    epochs = train_args.get('epochs', '未知')
    batch = train_args.get('batch', '未知')
    device = train_args.get('device', '未知')
    data_yaml = train_args.get('data', '未知')
    
    # ========== 3. 打印详细信息 ==========
    print("\n" + "="*60)
    print("YOLO11 训练模型信息汇总")
    print("="*60)
    
    print(f"\n【模型文件】: {p.absolute()}")
    print(f"【模型变体】: {arch}")
    print(f"【输入尺寸】: {imgsz}×{imgsz} 像素 (imgsz={imgsz})")
    
    print(f"\n【超参数 (hyp)】:")
    if hyp:
        for k, v in hyp.items():
            print(f"  {k:15}: {v}")
    else:
        print("  未找到超参数信息（可能存储在 train_args 的其他位置）")
    
    print(f"\n【其他训练设置】:")
    print(f"  训练轮数 (epochs): {epochs}")
    print(f"  批次大小 (batch) : {batch}")
    print(f"  训练设备 (device) : {device}")
    print(f"  数据集配置路径  : {data_yaml}")
    
    # 可选的类别数及类别名
    if 'names' in train_args:
        names = train_args['names']
        print(f"  类别数 (nc)      : {len(names)}")
        print(f"  类别名          : {list(names.values())}")
    elif hasattr(model, 'names'):
        print(f"  类别数 (nc)      : {len(model.names)}")
        print(f"  类别名          : {list(model.names.values())}")
    
    # 模型结构简要信息（层数、参数数量）
    if hasattr(model, 'model') and hasattr(model.model, 'model'):
        try:
            summary = model.model.model[-1]  # 粗略
            print(f"\n【模型结构摘要】")
            print(f"  总层数: {len(model.model.model)}")
            # 参数数量（百万）
            n_params = sum(p.numel() for p in model.model.parameters())
            print(f"  参数量: {n_params/1e6:.2f} M")
        except:
            pass
    
    print("\n" + "="*60)
    
except Exception as e:
    print(f"加载模型或提取信息时出错：{e}")
    import traceback
    traceback.print_exc()